# Parametric Curve Parameter Estimation

Find the unknown parameters `theta`, `M`, and `X` from the given `(x,y)` points.


## 1. Import libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution


## 2. Load the dataset


In [ ]:
df = pd.read_csv('data/xy_data.csv')
df.head()


In [ ]:
print('Shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())


## 3. Visualize the given points


In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(df['x'], df['y'], s=8)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Given XY Data')
plt.grid(True)
plt.show()


## 4. Define the parametric equation

The assignment gives:

$x(t)=t\cos(\theta)-e^{M|t|}\sin(0.3t)\sin(\theta)+X$

$y(t)=42+t\sin(\theta)+e^{M|t|}\sin(0.3t)\cos(\theta)$

with $0<\theta<50^\circ$, $-0.05<M<0.05$, $0<X<100$, and $6<t<60$.


In [ ]:
def curve(t, theta_deg, M, X):
    theta = np.deg2rad(theta_deg)
    amplitude = np.exp(M * np.abs(t)) * np.sin(0.3 * t)
    x = t*np.cos(theta) - amplitude*np.sin(theta) + X
    y = 42 + t*np.sin(theta) + amplitude*np.cos(theta)
    return x, y


## 5. Recover `t` from an observed `(x,y)` point

Rearranging the two equations gives:

$t=(x-X)\cos(\theta)+(y-42)\sin(\theta)$


In [ ]:
def calculate_t(x, y, theta_deg, X):
    theta = np.deg2rad(theta_deg)
    return ((x-X)*np.cos(theta) + (y-42)*np.sin(theta))


## 6. Define the L1 objective

For each candidate `(theta, M, X)`, infer `t`, generate the corresponding curve point, and calculate the mean L1 error:

$L1=mean(|x-x_{pred}|+|y-y_{pred}|)$


In [ ]:
x_data = df['x'].values
y_data = df['y'].values

def l1_loss(params):
    theta, M, X = params
    t = calculate_t(x_data, y_data, theta, X)

    if np.any(t <= 6) or np.any(t >= 60):
        return 1e6

    xp, yp = curve(t, theta, M, X)
    return np.mean(np.abs(x_data-xp) + np.abs(y_data-yp))


## 7. Optimize the unknown parameters


In [ ]:
bounds = [
    (1e-6, 49.999999),
    (-0.049999, 0.049999),
    (1e-6, 99.999999)
]

result = differential_evolution(
    l1_loss,
    bounds,
    seed=42,
    popsize=20,
    maxiter=1000,
    tol=1e-10,
    polish=True
)

theta, M, X = result.x

print('theta =', theta)
print('M     =', M)
print('X     =', X)
print('L1    =', result.fun)


## 8. Plot the final curve


In [ ]:
t_plot = np.linspace(6, 60, 5000)
x_curve, y_curve = curve(t_plot, theta, M, X)

plt.figure(figsize=(10,6))
plt.scatter(x_data, y_data, s=8, label='Given data')
plt.plot(x_curve, y_curve, linewidth=2, label='Fitted curve')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Given Data vs Fitted Parametric Curve')
plt.legend()
plt.grid(True)
plt.show()


## 9. Final result

The supplied dataset gives:

**theta = 30°**  
**M = 0.03**  
**X = 55**
